In [11]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Conectar a MongoDB
client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_libros']

# Crear SparkSession
spark = SparkSession.builder \
    .appName("G1_Ecommerce_Mystery") \
    .getOrCreate()

print("✓ Spark inicializado")

Scrapeando Mystery books...
Libros encontrados: 20

✓ 20 libros insertados en MongoDB


In [13]:
# Obtener datos de MongoDB
datos = list(coleccion.find({"grupo": "G1_Ecommerce"}))
print(f"✓ {len(datos)} registros en MongoDB")

# Limpiar: remover _id que Spark no entiende
datos_limpios = [{k: v for k, v in doc.items() if k != '_id'} for doc in datos]

# Convertir a DataFrame de Spark
df = spark.createDataFrame(datos_limpios)

print(f"✓ DataFrame: {df.count()} registros")

✓ Total registros de G1_Ecommerce: 40

Primer registro:
{'_id': ObjectId('69ddb1a3b2cbc1eee1f30721'), 'item': 'Sharp Objects', 'valor': 47.82, 'categoria': 'Mystery', 'grupo': 'G1_Ecommerce', 'fecha_captura': '2026-04-14 03:16:51'}


In [15]:
print("\nEsquema de datos:")
df.printSchema()

print("\nPrimeras 5 filas:")
df.show(5, truncate=False)

✓ Spark inicializado
✓ 40 registros en MongoDB
✓ DataFrame: 40 registros
+---------+-------------------+------------+--------------------+-----+
|categoria|      fecha_captura|       grupo|                item|valor|
+---------+-------------------+------------+--------------------+-----+
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|       Sharp Objects|47.82|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|In a Dark, Dark Wood|19.63|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce| The Past Never Ends| 56.5|
+---------+-------------------+------------+--------------------+-----+
only showing top 3 rows



In [12]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("item").isNotNull()) & (col("valor") > 0)
)

print("\nLIMPIEZA COMPLETADA")
print(f"   Registros originales: {df.count()}")
print(f"   Registros válidos: {df_filtrado.count()}")

✓ Spark inicializado
✓ 40 registros en MongoDB


TypeError: Unable to infer the type of the field _id.

In [ ]:
print("\na) Selección de columnas:")
df.select("item", "valor").show(5)

print("\nb) Productos que cuestan más de £15:")
df.filter(df["valor"] > 15).show(5)

print("\nc) Cantidad de productos por grupo:")
df.groupBy("grupo").count().show()

In [5]:
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("\nANÁLISIS DE MERCADO POR CATEGORÍA:")
reporte_categorias.show()

CONEXIÓN EXITOSA: Listo para insertar datos de Mystery


In [6]:
ganador = df.orderBy(F.desc("valor")).limit(1)

print("\nEL DETECTIVE DE PRECIOS:")
print("=" * 60)
ganador.select("item", "valor", "categoria", "grupo", "fecha_captura").show(truncate=False)

# Extraer valores para mostrar formateado
resultado = ganador.select("item", "valor", "categoria", "fecha_captura").collect()[0]

print("\nPRODUCTO MÁS CARO:")
print(f"   Nombre: {resultado['item']}")
print(f"   Precio: £{resultado['valor']}")
print(f"   Categoría: {resultado['categoria']}")
print(f"   Fecha captura: {resultado['fecha_captura']}")
print("=" * 60)

✓ Spark listo


In [7]:
ticket_salida = df.filter(F.col("grupo") == "G1_Ecommerce") \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"\n--- TICKET DE SALIDA: G1_Ecommerce ---")
ticket_salida.show()

Iniciando sesión de Spark...
Spark conectado a MongoDB


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Iniciando sesión de Spark...")

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", 
            "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", 
            "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

# Cargar datos desde MongoDB
df = spark.read.format("mongodb").load()

print("Spark conectado a MongoDB")
print(f"Registros cargados: {df.count()}")

Iniciando sesión de Spark...


Py4JJavaError: An error occurred while calling o36.load.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: mongodb. Please find packages at `https://spark.apache.org/third-party-projects.html`.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:724)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:647)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:697)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:208)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:172)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at j

In [ ]:
# Esto nos dice qué columnas entendió Spark
print("Esquema de datos:")
df.printSchema()

# Mostrar las primeras 5 filas
print("\nPrimeras 5 filas:")
df.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Solo queremos registros válidos
df_filtrado = df.filter(
    (col("item").isNotNull()) & (col("valor") > 0)
)

print("LIMPIEZA COMPLETADA")
print(f"   Registros originales: {df.count()}")
print(f"   Registros válidos: {df_filtrado.count()}")

In [ ]:
# a) Selección simple
print("Selección de columnas:")
df.select("item", "valor").show(5)

# b) Filtrado
print("\nProductos que cuestan más de £50:")
df.filter(df["valor"] > 50).show(5)

# c) Agregación por grupo
print("\nCantidad de productos por grupo:")
df.groupBy("grupo").count().show()

In [ ]:
# Agregación completa por CATEGORIA
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("\nANÁLISIS DE MERCADO POR CATEGORÍA:")
reporte_categorias.show()

In [ ]:
# Ordenar por valor de forma descendente y tomar el primero
ganador = df.orderBy(F.desc("valor")).limit(1)

print("\nEL DETECTIVE DE PRECIOS:")
print("=" * 60)
ganador.select("item", "valor", "categoria", "grupo", "fecha_captura").show(truncate=False)

# Extraer los valores para mostrar de forma más legible
resultado = ganador.select("item", "valor", "categoria", "fecha_captura").collect()[0]

print("\nPRODUCTO MÁS CARO:")
print(f"   Nombre: {resultado['item']}")
print(f"   Precio: £{resultado['valor']}")
print(f"   Categoría: {resultado['categoria']}")
print(f"   Fecha captura: {resultado['fecha_captura']}")
print("=" * 60)